In [2]:
import pandas as pd

from eavs.config import RAW_DATA_DIR
from eavs.config import CLEANED_DATA_DIR

2026-06-09 22:25:14.302 | INFO     | eavs.config:<module>:11 - PROJ_ROOT path is: D:\ruggbk\GitHub\eavs_clc


## Load CVAP data and perform basic EDA

In [3]:
cvap_2024: pd.DataFrame = pd.read_csv(
    RAW_DATA_DIR / "cvap" / "CVAP_2020-2024_ACS_csv_files" / "County.csv"
)

cvap_2024.head()

,geoname,lntitle,geoid,lnnumber,tot_est,tot_moe,adu_est,adu_moe,cit_est,cit_moe,cvap_est,cvap_moe
0,"Autauga County, Alabama",Total,0500000US01001,1,59947,0,45916,75,59123,351,45205,273
1,"Autauga County, Alabama",Not Hispanic or Latino,0500000US01001,2,57661,0,44377,75,57173,322,44001,234
2,"Autauga County, Alabama",American Indian or Alaska Native Alone,0500000US01001,3,38,34,36,34,35,34,35,34
3,"Autauga County, Alabama",Asian Alone,0500000US01001,4,697,219,585,151,609,203,494,166
4,"Autauga County, Alabama",Black or African American Alone,0500000US01001,5,12486,299,9025,241,12475,299,9023,241


In [4]:
# Check filtering for Pennsylvania and summarize data
cvap_PA = cvap_2024[cvap_2024['geoname'].str.contains('Pennsylvania')]
print(f'Rows and columns: {cvap_PA.shape}')
print(f'Number of unique counties (geoids): {cvap_PA["geoid"].nunique()}')
print(f'Amount of null values: \n{cvap_PA.isnull().sum()}')


Rows and columns: (871, 12)
Number of unique counties (geoids): 67
Amount of null values: 
geoname     0
lntitle     0
geoid       0
lnnumber    0
tot_est     0
tot_moe     0
adu_est     0
adu_moe     0
cit_est     0
cit_moe     0
cvap_est    0
cvap_moe    0
dtype: int64


In [5]:
cvap_PA[['lnnumber', 'lntitle']].drop_duplicates().sort_values('lnnumber')

,lnnumber,lntitle
29198,1,Total
29199,2,Not Hispanic or Latino
29200,3,American Indian or Alaska Native Alone
29201,4,Asian Alone
29202,5,Black or African American Alone
29203,6,Native Hawaiian or Other Pacific Islander Alone
29204,7,White Alone
29205,8,American Indian or Alaska Native and White
29206,9,Asian and White
29207,10,Black or African American and White


In [6]:
cvap_PA.columns

Index(['geoname', 'lntitle', 'geoid', 'lnnumber', 'tot_est', 'tot_moe',
       'adu_est', 'adu_moe', 'cit_est', 'cit_moe', 'cvap_est', 'cvap_moe'],
      dtype='object')

## EAVS Data — Pennsylvania

Load the cleaned combined EAVS dataset and filter to Pennsylvania counties.

In [7]:
eavs = pd.read_parquet(CLEANED_DATA_DIR / "eavs_combined_cleaned.parquet")
eavs_pa = eavs[eavs['state_abbr'] == 'PA'].copy()

print(f'Shape: {eavs_pa.shape}')
print(f'Years: {sorted(eavs_pa["year"].unique())}')
print(f'Unique counties: {eavs_pa["fips_code"].nunique()}')
eavs_pa.head()

Shape: (201, 146)
Years: [np.int64(2020), np.int64(2022), np.int64(2024)]
Unique counties: 67


,jurisdiction_name,state,state_abbr,registered_eligible_voters,active_voters,inactive_voters,total_registrations_received,new_valid_registrations,pre_registrations,duplicate_registrations,...,provisional_ballots_rejected_wrong_jurisdiction,provisional_ballots_rejected_wrong_precinct,provisional_ballots_rejected_no_id,provisional_ballots_rejected_incomplete,provisional_ballots_rejected_ballot_missing,provisional_ballots_rejected_no_signature,provisional_ballots_rejected_non_matching_signature,provisional_ballots_rejected_already_voted,mail_ballots_successfully_cured,mail_ballots_unsuccessfully_cured
3515,ADAMS COUNTY,PENNSYLVANIA,PA,71667,66078,5589,28376,9762,0,3384,...,None,None,None,None,None,None,None,None,None,None
3516,ALLEGHENY COUNTY,PENNSYLVANIA,PA,937910,865734,72176,400538,88481,0,36367,...,None,None,None,None,None,None,None,None,None,None
3517,ARMSTRONG COUNTY,PENNSYLVANIA,PA,44676,41612,3064,16606,4404,0,1874,...,None,None,None,None,None,None,None,None,None,None
3518,BEAVER COUNTY,PENNSYLVANIA,PA,116346,109016,7330,44723,9662,0,6068,...,None,None,None,None,None,None,None,None,None,None
3519,BEDFORD COUNTY,PENNSYLVANIA,PA,34185,31537,2648,12143,3296,0,65,...,None,None,None,None,None,None,None,None,None,None


In [8]:
# Identify turnout-relevant columns
eavs_pa.columns.tolist()

['jurisdiction_name',
 'state',
 'state_abbr',
 'registered_eligible_voters',
 'active_voters',
 'inactive_voters',
 'total_registrations_received',
 'new_valid_registrations',
 'pre_registrations',
 'duplicate_registrations',
 'rejected_registrations',
 'intrajurisdiction_registration_updates',
 'interjurisdiction_registration_updates',
 'total_forms_mail_fax_email',
 'new_registrations_mail_fax_email',
 'duplicate_registrations_mail_fax_email',
 'rejected_registrations_mail_fax_email',
 'total_forms_in_person',
 'new_registrations_in_person',
 'duplicate_registrations_in_person',
 'rejected_registrations_in_person',
 'total_forms_online',
 'new_registrations_online',
 'duplicate_registrations_online',
 'rejected_registrations_online',
 'total_forms_dmv',
 'new_registrations_dmv',
 'duplicate_registrations_dmv',
 'rejected_registrations_dmv',
 'total_forms_mandatory_nvra',
 'new_registrations_mandatory_nvra',
 'duplicate_registrations_mandatory_nvra',
 'rejected_registrations_mandator

## Merge EAVS and CVAP (Pennsylvania only)
Add estimate of CVAP population (`cvap_est`) per demographic (`lntitle`)

In [9]:
cvap_pivot = (
    cvap_PA.pivot_table(index=["geoid", "geoname"], columns="lntitle", values="cvap_est")
    .reset_index()
)
cvap_pivot.columns.name = None
cvap_pivot["geoid"] = cvap_pivot["geoid"].str.replace("0500000US", "")
cvap_pivot.head()

,geoid,geoname,American Indian or Alaska Native Alone,American Indian or Alaska Native and Black or African American,American Indian or Alaska Native and White,Asian Alone,Asian and White,Black or African American Alone,Black or African American and White,Hispanic or Latino,Native Hawaiian or Other Pacific Islander Alone,Not Hispanic or Latino,Remainder of Two or More Race Responses,Total,White Alone
0,42001,"Adams County, Pennsylvania",17.0,1.0,355.0,367.0,141.0,1279.0,447.0,4192.0,10.0,79182.0,18.0,83375.0,76551.0
1,42003,"Allegheny County, Pennsylvania",452.0,1941.0,2392.0,23994.0,4085.0,113007.0,10086.0,21977.0,165.0,949834.0,2220.0,971813.0,791481.0
2,42005,"Armstrong County, Pennsylvania",9.0,28.0,176.0,120.0,29.0,441.0,193.0,352.0,1.0,51796.0,16.0,52146.0,50782.0
3,42007,"Beaver County, Pennsylvania",39.0,66.0,439.0,564.0,314.0,7623.0,1007.0,2373.0,10.0,130730.0,279.0,133101.0,120392.0
4,42009,"Bedford County, Pennsylvania",21.0,4.0,144.0,114.0,45.0,211.0,116.0,311.0,0.0,37717.0,20.0,38027.0,37032.0


In [10]:
cvap_pivot.describe().loc["mean"].sort_values(ascending=False)

Total                                                             148683.671642
Not Hispanic or Latino                                            139403.776119
White Alone                                                       117773.731343
Black or African American Alone                                    14784.925373
Hispanic or Latino                                                  9279.850746
Asian Alone                                                         3977.582090
Black or African American and White                                 1165.044776
Asian and White                                                      595.223881
American Indian or Alaska Native and White                           483.716418
Remainder of Two or More Race Responses                              352.671642
American Indian or Alaska Native and Black or African American       160.059701
American Indian or Alaska Native Alone                                81.462687
Native Hawaiian or Other Pacific Islande

In [11]:
PA_df = (
    eavs_pa.loc[eavs_pa['year'] == 2024]
    .merge(cvap_pivot, left_on='fips_code', right_on='geoid', how='left')
    .drop(columns='geoid')
)
PA_df.head()

,jurisdiction_name,state,state_abbr,registered_eligible_voters,active_voters,inactive_voters,total_registrations_received,new_valid_registrations,pre_registrations,duplicate_registrations,...,Asian Alone,Asian and White,Black or African American Alone,Black or African American and White,Hispanic or Latino,Native Hawaiian or Other Pacific Islander Alone,Not Hispanic or Latino,Remainder of Two or More Race Responses,Total,White Alone
0,ADAMS COUNTY,PENNSYLVANIA,PA,76008,70087,5921,28929,8477,Data not available,4816,...,367.0,141.0,1279.0,447.0,4192.0,10.0,79182.0,18.0,83375.0,76551.0
1,ALLEGHENY COUNTY,PENNSYLVANIA,PA,952543,864332,88211,401572,87191,Data not available,34331,...,23994.0,4085.0,113007.0,10086.0,21977.0,165.0,949834.0,2220.0,971813.0,791481.0
2,ARMSTRONG COUNTY,PENNSYLVANIA,PA,44671,42263,2408,17129,4134,Data not available,2269,...,120.0,29.0,441.0,193.0,352.0,1.0,51796.0,16.0,52146.0,50782.0
3,BEAVER COUNTY,PENNSYLVANIA,PA,117448,109578,7870,41533,8227,Data not available,5006,...,564.0,314.0,7623.0,1007.0,2373.0,10.0,130730.0,279.0,133101.0,120392.0
4,BEDFORD COUNTY,PENNSYLVANIA,PA,33801,31862,1939,11464,2823,Data not available,679,...,114.0,45.0,211.0,116.0,311.0,0.0,37717.0,20.0,38027.0,37032.0


In [12]:
eavs_pa.shape

(201, 146)

In [13]:
PA_df.columns.tolist()

['jurisdiction_name',
 'state',
 'state_abbr',
 'registered_eligible_voters',
 'active_voters',
 'inactive_voters',
 'total_registrations_received',
 'new_valid_registrations',
 'pre_registrations',
 'duplicate_registrations',
 'rejected_registrations',
 'intrajurisdiction_registration_updates',
 'interjurisdiction_registration_updates',
 'total_forms_mail_fax_email',
 'new_registrations_mail_fax_email',
 'duplicate_registrations_mail_fax_email',
 'rejected_registrations_mail_fax_email',
 'total_forms_in_person',
 'new_registrations_in_person',
 'duplicate_registrations_in_person',
 'rejected_registrations_in_person',
 'total_forms_online',
 'new_registrations_online',
 'duplicate_registrations_online',
 'rejected_registrations_online',
 'total_forms_dmv',
 'new_registrations_dmv',
 'duplicate_registrations_dmv',
 'rejected_registrations_dmv',
 'total_forms_mandatory_nvra',
 'new_registrations_mandatory_nvra',
 'duplicate_registrations_mandatory_nvra',
 'rejected_registrations_mandator

In [19]:
cols_to_check = [
    "new_registrations_online", "new_registrations_dmv", "new_registrations_in_person",
    "new_registrations_mail_fax_email", "new_registrations_advocacy_groups",
    "mail_ballots_rejected_total", "mail_ballots_counted",
    "provisional_ballots_rejected_total", "provisional_ballots_cast_total",
]
for col in cols_to_check:
    n_nulls = pd.to_numeric(PA_df[col], errors="coerce").isna().sum()
    print(f"{col}: {n_nulls} missing out of {len(PA_df)}")



new_registrations_online: 0 missing out of 67
new_registrations_dmv: 67 missing out of 67
new_registrations_in_person: 0 missing out of 67
new_registrations_mail_fax_email: 0 missing out of 67
new_registrations_advocacy_groups: 67 missing out of 67
mail_ballots_rejected_total: 0 missing out of 67
mail_ballots_counted: 0 missing out of 67
provisional_ballots_rejected_total: 0 missing out of 67
provisional_ballots_cast_total: 0 missing out of 67
